## Trabajo Práctico Especial: Global Food & Nutrition Database

**Integrantes:** [Nombre y Apellido 1, Nombre y Apellido 2, Nombre y Apellido 3]  
**Ayudante Asignado:** Duilio Deangeli  

---

### Estructura de la Entrega y Criterio Metodológico
Siguiendo las pautas de la cátedra y el principio rector donde **"El tipo de variable gobierna el análisis"**, este trabajo se organiza en cinco secciones clave que se corresponden directamente con el informe final:

1. **Sección 1: Carga y Diagnóstico Exploratorio Inicial (EDA):** Origen de datos (Open Food Facts / nutricional), taxonomía de variables, distribuciones, valores atípicos y patrones de valores ausentes.
2. **Sección 2: Estrategia de Limpieza y Preprocesamiento:** Tratamiento justificado de nulos, duplicados, inconsistencias nutricionales y tipos de datos.
3. **Sección 3: Validación de Hipótesis de la Cátedra:**
   - **Hipótesis A (Univariada):** Proporción de productos altos en grasas saturadas (> 5g/100g) > 25%.
   - **Hipótesis B (Bivariada):** Grado de procesamiento industrial (NOVA) vs Calificación Nutri-Score (ordinal vs ordinal).
   - **Hipótesis C (Multivariada):** Capacidad explicativa conjunta de NOVA, categoría y alérgenos sobre Nutri-Score desfavorable (D/E).
4. **Sección 4: Planteo y Validación de Hipótesis Propias:** Formulación con sustento de dominio y contrastación empírica (Univariada, Bivariada y Multivariada).
5. **Sección 5: Conclusiones y Discusión Final:** Síntesis orientada a la toma de decisiones y respuesta al *"So What?"*.

## 0. Configuración del Entorno y Librerías

In [3]:

import pandas as pd


## 1. Carga del Dataset y Primer Diagnóstico

Cargamos el archivo `global_food_nutrition.csv` preservando una copia inmutable del dataframe crudo (`df_raw`).

In [4]:
DATASET_PATH = "global_food_nutrition.csv"

df_raw = pd.read_csv(DATASET_PATH)
print(f"Dimensiones iniciales: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas\n")
df_raw.head()

Dimensiones iniciales: 4997 filas × 24 columnas



,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged


In [ ]:
# Información general de tipos y completitud
df_raw.info()

In [ ]:
# Resumen de variables numéricas cuantitativas
df_raw.describe()

### Clasificación Inicial de Variables (Taxonomía)

| Variable | Tipo de Variable | Escala de Medición | Observaciones / Unidad |
| :--- | :--- | :--- | :--- |
| `product_name`, `brands` | Cualitativa | Nominal | Identificadores de producto y marca |
| `categories` | Cualitativa | Nominal multivalor | Categorías jerárquicas (separadas por comas) |
| `ingredients` | Cualitativa | Nominal texto libre | Listado textual de ingredientes |
| `nutriscore_grade` | Cualitativa | Ordinal | Letras A < B < C < D < E |
| `nova_group` | Cualitativa / Discreta | Ordinal | Niveles 1, 2, 3, 4 (Grado de procesamiento) |
| `ecoscore_grade` | Cualitativa | Ordinal / Nominal | Impacto ambiental |
| `allergens` | Cualitativa | Nominal multivalor | Alérgenos declarados |
| `contains_*` (gluten, dairy, etc.) | Cualitativa | Binaria (Booleana) | Flags de alérgenos comunes |
| `energy_kcal` | Cuantitativa | Continua | Kilocalorías por 100g |
| `fat_100g`, `saturated_fat_100g` | Cuantitativa | Continua | Gramos por 100g |
| `carbs_100g`, `sugars_100g` | Cuantitativa | Continua | Gramos por 100g |
| `fiber_100g`, `proteins_100g` | Cuantitativa | Continua | Gramos por 100g |
| `salt_100g`, `sodium_100g` | Cuantitativa | Continua | Gramos por 100g |
| `food_type` | Cualitativa | Nominal | Tipo de producto (ej. Branded/Packaged) |

## 2. Limpieza y Preprocesamiento de Datos

> En esta sección documentamos e implementamos: control de duplicados, saneamiento de tipos, imputación o filtrado fundamentado de valores nulos e inconsistencias biológicas/físicas (por ejemplo valores nutricionales negativos o sumas de macronutrientes > 100g).

## 3. Validación de Hipótesis de la Cátedra

### Hipótesis A (Univariada)
> *"La proporción de productos catalogados como 'altos en grasas saturadas' (> 5 g de grasa saturada por cada 100 g de producto) en la población de referencia es superior al 25%."*

### Hipótesis B (Bivariada)
> *"El grado de procesamiento industrial de un alimento está asociado con la calificación nutricional Nutri-Score que recibe, de manera que los productos más procesados presentan calificaciones más desfavorables."*

### Hipótesis C (Multivariada)
> *"El grado de procesamiento industrial (NOVA), la categoría del producto y la cantidad de alérgenos declarados explican conjuntamente la probabilidad de que un producto reciba una calificación Nutri-Score desfavorable (D/E), sin utilizar directamente su composición nutricional."*

## 4. Planteo y Validación de Hipótesis Propias

### Hipótesis 1 (Univariada)
### Hipótesis 2 (Bivariada)
### Hipótesis 3 (Multivariada)

## 5. Conclusiones y Discusión Final ("So What?")